In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import statsmodels
from statsmodels.tsa.seasonal import MSTL
from statsmodels.graphics.tsaplots import plot_acf
import pandas as pd
import os
import glob
import pandas as pd

In [4]:
SALES_PATH = "../data/processed/*.parquet"

In [5]:
files = glob.glob(SALES_PATH)
print(files)

df = pd.concat([pd.read_parquet(file,filters=[("year",">=",2015),("year","<=",2016)]) for file in files], ignore_index=True)

['../data/processed/part_0001.parquet', '../data/processed/part_0011.parquet', '../data/processed/part_0018.parquet', '../data/processed/part_0008.parquet', '../data/processed/part_0027.parquet', '../data/processed/part_0009.parquet', '../data/processed/part_0019.parquet', '../data/processed/part_0026.parquet', '../data/processed/part_0010.parquet', '../data/processed/part_0000.parquet', '../data/processed/part_0024.parquet', '../data/processed/part_0002.parquet', '../data/processed/part_0012.parquet', '../data/processed/part_0013.parquet', '../data/processed/part_0003.parquet', '../data/processed/part_0025.parquet', '../data/processed/part_0020.parquet', '../data/processed/part_0030.parquet', '../data/processed/part_0029.parquet', '../data/processed/part_0006.parquet', '../data/processed/part_0016.parquet', '../data/processed/part_0028.parquet', '../data/processed/part_0017.parquet', '../data/processed/part_0007.parquet', '../data/processed/part_0021.parquet', '../data/processed/part_

In [6]:
# testing connection and reading data
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns",None)

con = duckdb.connect()

df = con.execute(f"""select * from read_parquet('{SALES_PATH}') where year>=2015 and year<=2016 order by date""").df()
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,item_id,dept_id,cat_id,store_id,state_id,date,wm_yr_wk,d,sales,sell_price,weekday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap
0,FOODS_2_174,FOODS_2,FOODS,CA_1,CA,2015-01-01,11448,1434,0,2.68,Thursday,1,2015,NewYear,National,No Event,No Event,1
1,FOODS_2_175,FOODS_2,FOODS,CA_1,CA,2015-01-01,11448,1434,5,2.48,Thursday,1,2015,NewYear,National,No Event,No Event,1
2,FOODS_2_176,FOODS_2,FOODS,CA_1,CA,2015-01-01,11448,1434,0,2.68,Thursday,1,2015,NewYear,National,No Event,No Event,1
3,FOODS_2_177,FOODS_2,FOODS,CA_1,CA,2015-01-01,11448,1434,0,1.49,Thursday,1,2015,NewYear,National,No Event,No Event,1
4,FOODS_2_178,FOODS_2,FOODS,CA_1,CA,2015-01-01,11448,1434,0,2.50,Thursday,1,2015,NewYear,National,No Event,No Event,1


In [10]:
df[df["d"]==1913][["date"]].max()

date   2016-04-24
dtype: datetime64[us]

In [14]:

from datetime import timedelta


N_FOLDS = 3
FORECAST_HORIZON=28

def make_folds(start_date, end_date) -> dict:

    folds = []

    for i in range(3, 0, -1):
        train_end = end_date - timedelta(days=i * FORECAST_HORIZON)
        valid_start = train_end + timedelta(days=1)
        valid_end = valid_start + timedelta(days=FORECAST_HORIZON - 1)

        folds.append(
            {
                "folds": N_FOLDS - i + 1,
                "train_start": start_date,
                "train_end": train_end,
                "valid_start": valid_start,
                "valid_end": valid_end,
            }
        )

    return folds

df = df[df["d"]<=1913]
folds = make_folds(df["date"].min(), df["date"].max())
for i, fold in enumerate(folds):
    print("train --> ", fold["train_start"], fold["train_end"])
    print("valid --> ", fold["valid_start"], fold["valid_end"])

folds[0]["valid_start"]

val3_start = folds[2]["valid_start"]
val3_end = folds[2]["valid_end"]
df_val_3 = df[(df["date"]>=val3_start) & (df["date"]>=val3_end)]
df_val_3


train -->  2015-01-01 00:00:00 2016-01-31 00:00:00
valid -->  2016-02-01 00:00:00 2016-02-28 00:00:00
train -->  2015-01-01 00:00:00 2016-02-28 00:00:00
valid -->  2016-02-29 00:00:00 2016-03-27 00:00:00
train -->  2015-01-01 00:00:00 2016-03-27 00:00:00
valid -->  2016-03-28 00:00:00 2016-04-24 00:00:00


,item_id,dept_id,cat_id,store_id,state_id,date,wm_yr_wk,d,sales,sell_price,weekday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap
14505269,FOODS_2_174,FOODS_2,FOODS,CA_1,CA,2016-04-24,11613,1913,3,2.98,Sunday,4,2016,No Event,No Event,No Event,No Event,0
14505270,FOODS_2_175,FOODS_2,FOODS,CA_1,CA,2016-04-24,11613,1913,4,2.48,Sunday,4,2016,No Event,No Event,No Event,No Event,0
14505271,FOODS_2_176,FOODS_2,FOODS,CA_1,CA,2016-04-24,11613,1913,1,2.98,Sunday,4,2016,No Event,No Event,No Event,No Event,0
14505272,FOODS_2_177,FOODS_2,FOODS,CA_1,CA,2016-04-24,11613,1913,1,3.33,Sunday,4,2016,No Event,No Event,No Event,No Event,0
14505273,FOODS_2_178,FOODS_2,FOODS,CA_1,CA,2016-04-24,11613,1913,0,2.50,Sunday,4,2016,No Event,No Event,No Event,No Event,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14535754,FOODS_3_330,FOODS_3,FOODS,WI_3,WI,2016-04-24,11613,1913,3,3.98,Sunday,4,2016,No Event,No Event,No Event,No Event,0
14535755,FOODS_3_331,FOODS_3,FOODS,WI_3,WI,2016-04-24,11613,1913,1,1.58,Sunday,4,2016,No Event,No Event,No Event,No Event,0
14535756,FOODS_3_332,FOODS_3,FOODS,WI_3,WI,2016-04-24,11613,1913,1,1.98,Sunday,4,2016,No Event,No Event,No Event,No Event,0
14535757,FOODS_3_333,FOODS_3,FOODS,WI_3,WI,2016-04-24,11613,1913,6,1.98,Sunday,4,2016,No Event,No Event,No Event,No Event,0
